In [ ]:
import os
import numpy as np
import torch
import random

import ray
ray.init(num_cpus=16)  # Usa todos los cores que quieras

# Set environment variable for Python hash seed
os.environ["PYTHONHASHSEED"] = "42"

# Set seeds for determinism
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)

# PyTorch deterministic settings
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from ray import tune
from ray.rllib.algorithms.ppo import PPOConfig
from multiagent_ppo import MultiAgentF110, get_env_config, setup_policies_and_config
from ray.tune.registry import register_env
from ray.rllib.policy.policy import PolicySpec

# Path to the results directory
PATH_RESULTS = os.path.abspath("./ray_results")

# Visualize results
print("Training complete. Results saved to:", PATH_RESULTS)
print("You can visualize the results using TensorBoard or Ray Dashboard.")
print(f"Note: To visualize results, run `tensorboard --logdir={PATH_RESULTS}` in your terminal.")

In [ ]:
# Register the environment
register_env("f1tenth_multi", lambda config: MultiAgentF110(config))

In [ ]:
# # Create temporary environment to get spaces and agents
# temp_env = MultiAgentF110(get_env_config())
# policies = {agent: PolicySpec(None, temp_env.observation_space, temp_env.action_space, {}) 
#             for agent in temp_env.agents}
# temp_env.close()

# # Configure PPO for multi-agent training with determinism
# config = (PPOConfig()
#           .environment("f1tenth_multi", env_config=get_env_config())
#           .framework("torch")
#           .api_stack(enable_rl_module_and_learner=False, enable_env_runner_and_connector_v2=False)
#           .env_runners(
#               num_env_runners=0, # Only use one environment runner
#               num_envs_per_env_runner=1,  # Ensure single environment per worker
#           )
#           .multi_agent(
#               policies=policies, 
#               policy_mapping_fn=lambda agent_id, *args, **kwargs: agent_id
#           )
#           .training(
#               train_batch_size=200
#           )
#           .evaluation(
#               evaluation_interval=10,
#               evaluation_num_env_runners=1,
#               evaluation_config={
#                   "seed": SEED + 1  # Different seed for evaluation
#               }
#           )
#           .debugging(
#               seed=SEED  # Additional seed setting
#           ))

# # Run training with Ray Tune
# tune.run(
#     "PPO",
#     config=config.to_dict(),
#     stop={"timesteps_total": 500_000},  # Train for 20,000 timesteps
#     checkpoint_freq=10,  # Save checkpoint every 10 iterations
#     storage_path=PATH_RESULTS,  # Use the path for results
#     name="f1tenth_multiagent_ppo",  # Experiment name 
# )

In [ ]:
import os
import numpy as np
import torch
import random
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from ray.rllib.algorithms.ppo import PPO as PPOTrainer, PPOConfig
from multiagent_ppo import MultiAgentF110, get_env_config
from ray.tune.registry import register_env
from ray.rllib.policy.policy import PolicySpec

# --- Determinismo ---
os.environ["PYTHONHASHSEED"] = "42"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --- Registrar el entorno ---
register_env("f1tenth_multi", lambda config: MultiAgentF110(config))

# --- Espacio de búsqueda de hiperparámetros ---
search_space = {
    "lr": tune.loguniform(1e-5, 1e-3),
    "gamma": tune.uniform(0.95, 0.999),
    "lambda_": tune.uniform(0.9, 1.0),
    "clip_param": tune.uniform(0.1, 0.4),
    "train_batch_size": tune.choice([256, 512, 1024, 2048, 4096]),
    "num_sgd_iter": tune.choice([5, 10, 20]),
    "sgd_minibatch_size": tune.choice([64, 128, 256]),
    "entropy_coeff": tune.uniform(0.0, 0.05),
    "model": {"fcnet_hiddens": tune.choice([(128, 128), (256, 256), (256, 128)])} 
}

def policy_dict():
    temp_env = MultiAgentF110(get_env_config())
    policies = {agent: PolicySpec(None, temp_env.observation_space, temp_env.action_space, {})
                for agent in temp_env.agents}
    temp_env.close()
    return policies

# --- Configuración base sin hiperparámetros ---
base_cfg = (
    PPOConfig()
    .environment("f1tenth_multi", env_config=get_env_config())
    .framework("torch")
    .api_stack(enable_rl_module_and_learner=False, enable_env_runner_and_connector_v2=False)
    .env_runners(num_env_runners=0, num_envs_per_env_runner=1)
    .multi_agent(
        policies=policy_dict(),
        policy_mapping_fn=lambda agent_id, episode, worker, **kwargs: agent_id
    )
    .evaluation(evaluation_interval=10, evaluation_num_env_runners=1, evaluation_config={"seed": SEED + 1})
    .debugging(seed=SEED)
).to_dict()

# --- Integrar espacio de búsqueda en la configuración ---
config = base_cfg.copy()
config.update(search_space)

# --- Buscador Bayesiano ---
default_metric = "env_runners/episode_reward_mean"  # <-- Cambia aquí
search_alg = OptunaSearch(metric=default_metric, mode="max", seed=SEED)

# --- Ejecutar tuning ---
tune.run(
    PPOTrainer,
    config=config,
    num_samples=50,
    stop={"timesteps_total": 200_000},
    checkpoint_freq=10,
    storage_path=PATH_RESULTS,
    name="f1tenth_multiagent_ppo_tune",
    search_alg=search_alg,
    metric=default_metric,   # <-- Añade esto para ser explícito
    mode="max"
)

In [ ]:
# from ray.tune import Analysis

# analysis = Analysis(PATH_RESULTS + "/f1tenth_multiagent_ppo_tune")
# print("Best hyperparameters found were: ", analysis.best_config(metric="episode_reward_mean", mode="max"))
# print("Best trial directory: ", analysis.best_logdir(metric="episode_reward_mean", mode="max"))

In [ ]:
# import glob, os
# from ray.rllib.algorithms.algorithm import Algorithm

# # --- 1. Find the latest checkpoint ---
# # Construct the path to the experiment directory
# exp_dir = os.path.join(PATH_RESULTS, "f1tenth_multiagent_ppo")

# # Find the latest trial directory within the experiment
# try:
#     latest_trial_dir = sorted(glob.glob(os.path.join(exp_dir, "PPO_*")))[-1]
#     print(f"Loading results from: {latest_trial_dir}")

#     # Find the latest checkpoint in that trial directory
#     latest_checkpoint_dir = sorted(glob.glob(os.path.join(latest_trial_dir, "checkpoint_*")))[-1]
#     print(f"Loading checkpoint from: {latest_checkpoint_dir}")
# except IndexError:
#     print(f"Error: No training results found in '{exp_dir}'. Please run the training cell first.")
#     # Exit gracefully if no checkpoint is found
#     latest_checkpoint_dir = None

# if latest_checkpoint_dir:
#     # --- 2. Restore the trained algorithm ---
#     algo = Algorithm.from_checkpoint(latest_checkpoint_dir)

#     # --- 3. Create environment for visualization ---
#     env_config = get_env_config()
#     env_config["render_mode"] = "human"  # Enable human-readable rendering
#     env = MultiAgentF110(env_config)

#     # --- 4. Run a simulation episode ---
#     print("Starting simulation... Close the simulation window to continue.")
#     obs, info = env.reset()
#     terminated = {"__all__": False}

#     while not terminated["__all__"]:
#         actions = {}
#         for agent_id, agent_obs in obs.items():
#             # Compute actions using the restored policy for each agent
#             actions[agent_id] = algo.compute_single_action(
#                 observation=agent_obs,
#                 policy_id=agent_id,
#                 explore=False  # Disable exploration for testing
#             )
        
#         # Step the environment with the computed actions
#         obs, reward, terminated, truncated, info = env.step(actions)
        
#         # Render the environment to visualize
#         env.render()

#     # --- 5. Clean up ---
#     env.close()
#     print("Simulation finished.")